# 🔀 Day 03a: Keras Functional API — Complex Architectures

---

## 🎯 What You'll Master Today
- Functional API vs Sequential — when to use which
- Multi-input / multi-output models
- Custom layers with `keras.layers.Layer`
- Subclassed Model with `keras.Model`

**Goal:** Build branching architectures and custom layers from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  Sequential vs Functional vs Subclassed                         ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Sequential:  Linear stack of layers. Simple, limited.          ║
║               model = Sequential([Dense, Dense, Dense])         ║
║                                                                   ║
║  Functional:  DAG of layers. Multi-input/output, skip conns.    ║
║               inputs = Input(shape)                              ║
║               x = Dense(64)(inputs)                              ║
║               model = Model(inputs, outputs)                    ║
║                                                                   ║
║  Subclassed:  Full Python control. Most flexible, least tools.  ║
║               class MyModel(keras.Model): ...                   ║
║                                                                   ║
║  Rule: Use Sequential when you can, Functional when you need,  ║
║        Subclass only for research.                               ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ============================================================
# 1. Functional API Basics
# ============================================================

# Step 1: Define inputs
inputs = keras.Input(shape=(10,), name='features')

# Step 2: Chain layers (call layers like functions)
x = layers.Dense(64, activation='relu')(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid', name='prediction')(x)

# Step 3: Create model
model = keras.Model(inputs=inputs, outputs=outputs, name='binary_classifier')

model.summary()

# Same compile/fit as Sequential
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("\n💡 Functional syntax: output_tensor = Layer()(input_tensor)")
print("   Each layer call creates a node in the computation graph")

In [ ]:
# ============================================================
# 2. Skip Connection (ResNet-style)
# ============================================================

def residual_block(x, units):
    """x → Dense → BN → ReLU → Dense → BN → Add(x) → ReLU"""
    shortcut = x
    
    x = layers.Dense(units)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dense(units)(x)
    x = layers.BatchNormalization()(x)
    
    # Skip connection: add input back
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

inputs = keras.Input(shape=(64,))
x = layers.Dense(64, activation='relu')(inputs)
x = residual_block(x, 64)
x = residual_block(x, 64)
outputs = layers.Dense(10, activation='softmax')(x)

resnet_model = keras.Model(inputs, outputs, name='mini_resnet')
resnet_model.summary()
print("\n💡 Skip connections can only be built with Functional API (not Sequential)")

In [ ]:
# ============================================================
# 3. Multi-Input / Multi-Output Model
# ============================================================

# Example: predict price AND category from features + text

# Input 1: numerical features
num_input = keras.Input(shape=(10,), name='numerical')
x1 = layers.Dense(32, activation='relu')(num_input)

# Input 2: text (as token sequences)
text_input = keras.Input(shape=(50,), dtype='int32', name='text')
x2 = layers.Embedding(input_dim=5000, output_dim=64)(text_input)
x2 = layers.GlobalAveragePooling1D()(x2)  # (batch, 50, 64) → (batch, 64)
x2 = layers.Dense(32, activation='relu')(x2)

# Merge
merged = layers.Concatenate()([x1, x2])  # (batch, 64)
shared = layers.Dense(64, activation='relu')(merged)
shared = layers.Dropout(0.3)(shared)

# Output 1: price regression
price_output = layers.Dense(1, name='price')(shared)

# Output 2: category classification
category_output = layers.Dense(5, activation='softmax', name='category')(shared)

multi_model = keras.Model(
    inputs=[num_input, text_input],
    outputs=[price_output, category_output],
    name='multi_task'
)

multi_model.compile(
    optimizer='adam',
    loss={'price': 'mse', 'category': 'sparse_categorical_crossentropy'},
    loss_weights={'price': 1.0, 'category': 0.5},  # weight the losses
    metrics={'price': 'mae', 'category': 'accuracy'},
)

multi_model.summary()
print("\n💡 Multi-output: pass dict to loss, loss_weights, metrics")

In [ ]:
# ============================================================
# 4. Custom Layer
# ============================================================

class ScaledDotProductAttention(layers.Layer):
    """Simple attention: score = Q @ K^T / sqrt(d), output = score @ V"""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    
    def build(self, input_shape):
        d = input_shape[-1]
        self.W_q = self.add_weight('W_q', shape=(d, self.units))
        self.W_k = self.add_weight('W_k', shape=(d, self.units))
        self.W_v = self.add_weight('W_v', shape=(d, self.units))
    
    def call(self, inputs):
        Q = inputs @ self.W_q
        K = inputs @ self.W_k
        V = inputs @ self.W_v
        
        d_k = tf.cast(self.units, tf.float32)
        scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(d_k)
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)
    
    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config

# Use custom layer in Functional API
inputs = keras.Input(shape=(20, 64))  # (seq_len, features)
attended = ScaledDotProductAttention(32)(inputs)
pooled = layers.GlobalAveragePooling1D()(attended)
outputs = layers.Dense(5, activation='softmax')(pooled)

attn_model = keras.Model(inputs, outputs)
print(f"Attention model output: {attn_model(tf.random.normal((2, 20, 64))).shape}")
print("\n💡 Custom layer: __init__ (config), build (create weights), call (forward)")
print("   get_config() needed for serialization")

In [ ]:
# ============================================================
# 5. Subclassed Model (like PyTorch nn.Module)
# ============================================================

class MyModel(keras.Model):
    def __init__(self, num_classes):
        super().__init__()
        self.dense1 = layers.Dense(64, activation='relu')
        self.bn = layers.BatchNormalization()
        self.dropout = layers.Dropout(0.3)
        self.dense2 = layers.Dense(num_classes)
    
    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.bn(x, training=training)  # BN behaves differently train vs eval
        x = self.dropout(x, training=training)  # Dropout only during training
        return self.dense2(x)

sub_model = MyModel(num_classes=3)

# Must call with data once to build
dummy = tf.random.normal((2, 10))
out = sub_model(dummy, training=False)
print(f"Subclassed model output: {out.shape}")

sub_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

print("""
╔══════════════════════════════════════════════════════╗
║  Model Type Comparison                              ║
╠══════════════════════════════════════════════════════╣
║  Sequential  │ Simple stacks only                   ║
║              │ model.summary() ✓, save ✓            ║
║  Functional  │ DAG, multi-IO, skip connections      ║
║              │ model.summary() ✓, save ✓            ║
║  Subclassed  │ Full Python, most flexible           ║
║              │ model.summary() limited, save harder ║
╚══════════════════════════════════════════════════════╝

💡 training=False is critical in call()!
   BatchNorm and Dropout behave differently at inference.
""")

---

## 🗺️ Functional API Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  KERAS FUNCTIONAL API                                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  inputs = Input(shape)                                           ║
║  x = Layer()(inputs)                                             ║
║  model = Model(inputs, outputs)                                  ║
║                                                                   ║
║  Merge layers: Concatenate, Add, Multiply                       ║
║  Skip connection: shortcut + layers.Add()([x, shortcut])        ║
║  Multi-output: dict for loss, metrics, loss_weights             ║
║                                                                   ║
║  Custom Layer:                                                   ║
║    __init__: store config                                        ║
║    build:    create weights (add_weight)                         ║
║    call:     forward pass logic                                  ║
║    get_config: for serialization                                 ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | When Functional over Sequential? | Multi-input/output, skip connections, shared layers, any non-linear topology |
| 2 | How to create a skip connection? | Store shortcut tensor, transform x through layers, then layers.Add()([x, shortcut]) |
| 3 | How to handle multi-output training? | Dict of losses per output name, loss_weights to balance them |
| 4 | Custom layer: build vs call? | build() creates weights (called once, sees input shape). call() is the forward pass (called every time) |
| 5 | Why pass training=False? | BN uses running stats at inference. Dropout is disabled at inference. Without it, predictions are wrong |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Functional: Input → Layer()(x) → Model(in, out)     │
## │  • Skip conn: Add()([x, shortcut])                      │
## │  • Multi-IO: list of inputs/outputs, dict of losses    │
## │  • Custom Layer: build(weights) + call(forward)         │
## │  • Always pass training arg for BN/Dropout              │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **tf.data** — efficient data pipelines